# 🧠 Mini-Batch Gradient Descent และการแปลงเป็นเวกเตอร์ (Vectorization)

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **Mini-Batch Gradient Descent**! ในสมุดบันทึกนี้ เราจะ:
1. เปรียบเทียบ Batch GD, Stochastic GD (SGD) และ Mini-Batch GD อย่างละเอียด
2. อิมพลีเมนต์ **vectorized Mini-Batch GD** จากศูนย์โดยใช้ NumPy
3. เปรียบเทียบเส้นทางการลู่เข้าสำหรับการเพิ่มประสิทธิภาพของทั้งสามวิธีบนแผนที่เส้นชั้นความสูงของลอสแบบ 2 มิติ (2D loss contour map)
4. แสดงภาพกราฟการลดลงของลอสเทียบกับขั้นตอนการอัปเดต
5. อธิบายการคูณเมทริกซ์บน GPU (vectorization) และรูปร่างของเทนเซอร์ (tensor shapes)
6. เชื่อมโยงพารามิเตอร์เหล่านี้กับอาร์กิวเมนต์ `batch` ของ YOLO และข้อผิดพลาด CUDA Out of Memory (OOM)

มาเริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันเลย

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. Simulating Data for Linear Regression

เราสร้างกลุ่มตัวอย่าง 100 ตัวตามกระบวนการเชิงเส้น $y = 3x + 1 + \text{noise}$

In [ ]:
n_samples = 100
X = np.random.rand(n_samples, 1)
y = 3 * X + 1 + np.random.normal(0, 0.15, (n_samples, 1))

## 2. การอิมพลีเมนต์ Vectorized Mini-Batch GD จากศูนย์ (from Scratch)

มาอิมพลีเมนต์ Mini-Batch GD กัน
ขั้นตอนการทำงานในแต่ละรอบ (epoch):
1. สลับดัชนีของข้อมูล (Shuffle indices) แบบสุ่ม
2. ดึงข้อมูลส่วนย่อย (Slices) ตามขนาด $B$ (batch size)
3. ทำการคำนวณผ่านไปข้างหน้า (Forward pass) และหาเกรเดียนต์ในรูปแบบเวกเตอร์ (Vectorized gradient computation) บน mini-batch:
   $$\text{dw} = \frac{1}{B} \mathbf{X}_{\text{batch}}^T (\hat{\mathbf{y}} - \mathbf{y}_{\text{batch}})$$
   $$\text{db} = \frac{1}{B} \sum (\hat{\mathbf{y}} - \mathbf{y}_{\text{batch}})$$
4. อัปเดตค่าน้ำหนักทันที

In [ ]:
def run_mini_batch_gd(X, y, batch_size=10, lr=0.1, epochs=10):
    w, b = 0.0, 0.0
    m = len(X)
    history = [[w, b]]
    
    for _ in range(epochs):
        indices = np.random.permutation(m)
        X_shuffled = X[indices]
        y_shuffled = y[indices]
        
        for i in range(0, m, batch_size):
            X_batch = X_shuffled[i : i + batch_size]
            y_batch = y_shuffled[i : i + batch_size]
            B = len(X_batch)
            
            preds = w * X_batch + b
            error = preds - y_batch
            
            # Vectorized gradients
            dw = (1 / B) * np.dot(X_batch.T, error)[0, 0]
            db = (1 / B) * np.sum(error)
            
            w -= lr * dw
            b -= lr * db
            history.append([float(w), float(b)])
            
    return np.array(history)

# Run optimization for all three modes
path_bgd = run_mini_batch_gd(X, y, batch_size=100, lr=0.1, epochs=50)
path_sgd = run_mini_batch_gd(X, y, batch_size=1, lr=0.01, epochs=2)
path_mbgd = run_mini_batch_gd(X, y, batch_size=10, lr=0.05, epochs=5)

## 3. การแสดงเส้นทางการลู่เข้าบนแผนที่เส้นชั้นความสูง (Contour Map)

ลองพล็อตเส้นทางการลู่เข้าของทั้งสามโมเดลบนแผนที่เส้นชั้นความสูงของพื้นที่ผิวของลอสแบบ 2 มิติ (2D Loss Landscape) แผ่นเดียวกัน

In [ ]:
# Compute loss grid
w_vals = np.linspace(-0.5, 4.5, 100)
b_vals = np.linspace(-0.5, 2.5, 100)
W, B = np.meshgrid(w_vals, b_vals)

Z = np.zeros_like(W)
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        w_tmp, b_tmp = W[i, j], B[i, j]
        Z[i, j] = (1 / (2 * n_samples)) * np.sum((w_tmp * X + b_tmp - y) ** 2)

# Plot paths
plt.figure(figsize=(10, 8))
contours = plt.contour(W, B, Z, levels=25, cmap='viridis')
plt.clabel(contours, inline=1, fontsize=8)

plt.plot(path_bgd[:, 0], path_bgd[:, 1], color='red', marker='o', linewidth=2.5, label='Batch GD (Size = 100)')
plt.plot(path_sgd[:, 0], path_sgd[:, 1], color='orange', alpha=0.6, linewidth=1.5, label='Stochastic GD (Size = 1)')
plt.plot(path_mbgd[:, 0], path_mbgd[:, 1], color='cyan', marker='x', linewidth=2, label='Mini-Batch GD (Size = 10)')

plt.scatter(3.0, 1.0, color='blue', s=120, marker='*', zorder=5, label='Target Minimum')
plt.xlabel('Weight (w)')
plt.ylabel('Bias (b)')
plt.title('Contour Map Comparison of Gradient Descent Variants')
plt.legend()
plt.show()

สังเกตได้ว่า:
-   **Batch GD:** ราบรื่นมากเป็นพิเศษ แต่อัปเดตเพียง 1 ครั้งต่อรอบ (epoch) เท่านั้น
-   **Stochastic GD:** มีสัญญาณรบกวนสูงมาก มีการส่ายไปมาอย่างไม่เป็นระเบียบจากด้านหนึ่งไปอีกด้านหนึ่ง
-   **Mini-Batch GD:** จุดกึ่งกลางที่เหมาะสมที่สุด: มันเดินตามเส้นทางที่เสถียรและมีระลอกคลื่นเพียงเล็กน้อย แต่อัปเดตบ่อยครั้ง ลู่เข้าหาเป้าหมายได้อย่างรวดเร็วและมีประสิทธิภาพมาก

## 💡 ความเชื่อมโยงกับ YOLO และการเรียนรู้เชิงลึก (Deep Learning)
*   **พารามิเตอร์ `batch`:** เมื่อเราฝึกโมเดล YOLO ขนาดของแบตช์จะถูกกำหนดด้วยอาร์กิวเมนต์ `batch`:
    ```bash
    yolo train model=yolo11n.pt data=data.yaml batch=16
    ```
*   **ข้อผิดพลาด CUDA Out of Memory (OOM):** หากเลือกขนาดแบตช์ที่ใหญ่เกินไป (เช่น `batch=64` หรือ `128` บน GPU ขนาดเล็ก) เทนเซอร์ขนาด `[64, 3, 640, 640]` ซึ่งบรรจุภาพและแผนผังผลลัพธ์ของเลเยอร์คอนโวลูชันระดับกลางทั้งหมดจะมีขนาดเกินขีดจำกัดหน่วยความจำของ GPU ทำให้เกิดการล่มจากข้อผิดพลาด CUDA OOM
*   **วิธีแก้ไข:** หากพบข้อผิดพลาด OOM ให้ลดขนาดแบตช์ลง (เช่น เปลี่ยนเป็น `batch=16` หรือ `batch=8` เป็นต้น)